## TOKENIZING WITH CODE

Tiktoken library only supports encodings from OpenAI models. To use tokenization from other models we need to follow different approach.

#### TOKENIZATION USING OPENAI TIKTOKEN LIBRARY

In [27]:
!uv pip install tiktoken

Using Python 3.14.3 environment at: d:\Private\LLM_Engineering\.venv
Checked 1 package in 10ms


In [28]:
import tiktoken

def count_tokens(text, model="gpt-4.1-mini"):
    encoding = tiktoken.encoding_for_model(model)
    return len(encoding.encode(text))

In [29]:
print(count_tokens("Hello, how are you?"))

6


In [30]:
def get_tokens(text, model="gpt-4.1-mini"):
    encoding = tiktoken.encoding_for_model(model)
    return encoding.encode(text)

In [31]:
tokens = get_tokens("Hello, how are you?")
for token in tokens:
    token_text = tiktoken.encoding_for_model("gpt-4.1-mini").decode([token])
    print(f"Token ID: {token}, Text: '{token_text}'")

Token ID: 13225, Text: 'Hello'
Token ID: 11, Text: ','
Token ID: 1495, Text: ' how'
Token ID: 553, Text: ' are'
Token ID: 481, Text: ' you'
Token ID: 30, Text: '?'


#### TOKENIZATION FOR OLLAMA MODELS (OPEN SOURCE)

Ollama doesn't support any tokenizer.

## THE ILLUSION OF 'MEMORY'

#### NOTE: 
Every call to an LLM is completely STATELESS. It's a totally new call, every single time. As AI engineers, it's our job to devise techniques to give the impression that the LLM has a "memory".

Example

In [61]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from IPython.display import Markdown, display

load_dotenv(override=True)

key = os.getenv("OLLAMA_API_KEY")
model = os.getenv("OLLAMA_MODEL")
url = os.getenv("OLLAMA_BASE_URL")

In [62]:
client = OpenAI(base_url=url, api_key=key)

response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hi! I'm Ujjwal"}
    ]
)
display(Markdown(response.choices[0].message.content))

Namaste Ujjwal! It's nice to meet you. How can I assist you today? Do you have any questions or topics you'd like to discuss?

In [63]:
client = OpenAI(base_url=url, api_key=key)

response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What's my name?"}
    ]
)
display(Markdown(response.choices[0].message.content))

I don't have any information about your personal identity, so I couldn't tell you what your name is without more context or some way of knowing who you are. I'm here to help with any question you may have, though! Is there something else on your mind that you'd like to chat about?

In [64]:
client = OpenAI(base_url=url, api_key=key)

response = client.chat.completions.create(
    model=model,
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hi! I'm Ujjwal"},
        {"role": "assistant", "content": "Hi Ujjwal! How can I assist you today?"},
        {"role": "user", "content": "What's my name?"}
    ]
)
display(Markdown(response.choices[0].message.content))

Your name is Ujjwal! I remember, we just had that conversation. Would you like to start something new?

#### NOTE:
1. Every call to an LLM is stateless,
2. We pass in the entire conversation so far in the input prompt, every time,
3. This gives the illusion that the LLM has memory - it apparently keeps the context of the conversation,
4. But this is a trick; it's a by-product of providing the entire conversation, every time,
5. An LLM just predicts the most likely next tokens in the sequence; if that sequence contains "My name is Ujjwal" and later "What's my name?" then it will predict.. Ujjwal!
7. The ChatGPT product uses exactly this trick - every time you send a message, it's the entire conversation that gets passed in,
8. "Does that mean we have to pay extra each time for all the conversation so far ?"
9. For sure it does. And that's what we WANT. We want the LLM to predict the next tokens in the sequence, looking back on the entire conversation. We want that compute to happen, so we need to pay the electricity bill for it!

#### MISCELLANEOUS

In [39]:
import ollama

response = ollama.chat(model="llama3.2:latest", messages=[
    {"role": "system", "content": "You are a helpful assistant."},
    {"role": "user", "content": "Hi! I'm Ujjwal"}
])

print(response["message"]["content"])


Hello Ujjwal! It's nice to meet you. Is there something I can help you with or would you like to chat?
